In [1]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

In [2]:

# some constant values
IMG_SIZE = (224, 224)
MODEL_PATH = 'meme.keras'
CLASS_NAMES = ['dipak', 'me', 'ram', 'shyam', 'unknown'] # Match the training classes


In [3]:

#loading the model
model = keras.models.load_model(MODEL_PATH)


In [4]:
# function for real-time camera inference
def camm(model):
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("cannot open camera")
        return

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        h, w, _ = frame.shape
        crop = frame[int(0.2*h):int(0.8*h), int(0.2*w):int(0.8*w)]

        # preprocessing the iamge
        img = cv2.resize(crop, IMG_SIZE)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = np.expand_dims(img, axis=0)
        img = preprocess_input(img)  # adding the same preprocessing as in training

        # running the prediction
        result = model.predict(img, verbose=0)[0]
        res_id = np.argmax(result)
        confidence = result[res_id] * 100
        predicted_class = CLASS_NAMES[res_id]
        # if the overall confidence is less than 60%, we consider it as unknown
        if predicted_class == "unknown" or confidence < 60:
            display_text = "Unknown"
            color = (0, 0, 255)  # red color for unknown
        else:
            display_text = f"{predicted_class} ({confidence:.1f}%)"
            color = (0, 255, 0)  # green color for other predictions

        # putting on the frame
        cv2.putText(
            frame,
            display_text,
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            color,
            2,
            cv2.LINE_AA
        )

        cv2.imshow("Palm Recognition", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()


In [5]:
camm(model)